In [ ]:
import os, sys
import json
import requests
import random
import string
from typing import Annotated, Dict
from operator import add
from langchain_groq import ChatGroq 
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command
from langchain_core.messages import BaseMessage,HumanMessage,SystemMessage,AIMessage,ToolMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, InjectedState, tools_condition
from dotenv import load_dotenv
from rich import print as rprint
sys.path.append(os.path.abspath("..")) 
from src.dialogue_classifier import classify_dialogue_act
from src.complaint_graph import complaint_flow

In [ ]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
def merge_dicts(left:dict, right:dict)->dict:

    if not right: return left
    if not left: return right

    def is_empty(v):
        return v is None or str(v).strip().lower() in ("none","null","")

    return {**left, **{k:v for k,v in right.items() if not is_empty(v)}}

class SupervisorState(MessagesState):
    """
        State for multi-agent system
    """   
    user_input: str
    user_intent: str

    actions_taken: str
    observations: Annotated[list, add]
    response: Annotated[list, add]
    
    complaint_data: Annotated[dict, merge_dicts]
    missing_info: list
    next: str
    customer_profile: Dict
    

In [ ]:
# Supervisor Node
def supervisor_node(state:SupervisorState)->SupervisorState:

    history = state.get("observations", [])
    user_input = state.get("user_input")
    current_intent = state.get("user_intent", "unknown")

    label, score = classify_dialogue_act(user_input)
    print("Label--->",label)
    print("Score--->",score)
    if score>=0.4:
        return Command(goto=label)
    else:
        return Command(goto="clarification_flow")
    
    

In [ ]:
test =  {"user_input": "My phone stopped working yesterday"}

supervisor_node(test)

In [ ]:
# @tool
def greeting_node(state:SupervisorState)->dict:

    """Greets the user warmly and sets the empathetic tone for the interaction."""
    
    user_input = state.get("user_input",[])
    prompt = f"""
    You are a helpful and empathetic Customer Success Assistant.
    Greet the user warmly based on their input: "{user_input}".
    """
    res = llm.invoke([SystemMessage(content=prompt)])
    print(res)
    
    return res.content

def clarification_node(state: SupervisorState) -> dict:
    """Refines vague input into a specific intent (Complaint, Retention, or Inquiry)."""
    
    user_input = state.get("user_input", "")
    
    system_prompt = """
    Role: Support Triage Agent
    Context: The user's input is too vague to route to a specific workflow.
    
    Task: Acknowledge the user's message and ask a single, polite question to clarify if they:
    - Have a technical problem/defect (Complaint)
    - Want to cancel, return, or switch services (Retention)
    - Need general info or instructions (Inquiry)
    
    Constraint: Be empathetic but direct. Maximum 25 words.
    """
    
    # Invoking the LLM with the specific user input
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_input)
    ])
    
    # Returning the response to update the message state
    return {"messages": [response]}

In [ ]:
# Worker nodes
def rag_node(state:SupervisorState)->SupervisorState:
    "Answers general user inquiries that fall outside the automated support ticket workflow."
    user_input=state.get("user_input",[])

    system_prompt="""
        You are a helpful support assistant. Provide clear, concise answers to the user's questions.
        """
    res = llm.invoke([SystemMessage(system_prompt)]+[HumanMessage(user_input)])
    
    return {'messages':res.content}

In [ ]:
apify_api_key = os.getenv("APIFY_API_KEY")

actor_id = "apift/walmart-scraper"
url = f"https://api.apify.com/v2/acts/{actor_id}/run-sync-get-dataset-items"

payload = {
    "searchTerms": ["4K TV"],
    "maxItems": 5,
    "maxPrice": 300
}

headers = {
  'Accept': 'application/json',
  'Authorization': 'Bearer ' + apify_api_key
}

response = requests.request("GET", url, headers=headers, data=payload)

print(response.text)

In [ ]:
def build_inquiry_node():
    graph = StateGraph(SupervisorState)
    
    graph.add_node("inquiry", rag_node)

    graph.add_edge(START, "inquiry")
    graph.add_edge("inquiry", END)

    memory = MemorySaver()
    return graph.compile(checkpointer=memory)

inquiry_graph = build_inquiry_node()

def inquiry_flow(state:SupervisorState)->SupervisorState:
    config = {"configurable": {"thread_id": "support_session_1"}} 
    return inquiry_graph.invoke(state, config=config)

In [ ]:
inquiry_graph

In [ ]:
user_input = {"user_input":"I need to know where is toothpaste."}

inquiry_flow(user_input)

In [ ]:
def extract_info_node(state:SupervisorState)->SupervisorState:
    """ 
    Extracts product, issue, and date from input into JSON.
    """
    user_input = state.get("user_input",[])

    system_prompt=f""" You are an entity extractor. 
    Your goal is to extract the product information, issue_type, and purchase_date 
    from the user's input.
    - product
    - issue_type
    - purchase_date

    Respond ONLY with a valid JSON object in this exact format:
    {{
        "product": string or null,
        "issue_type": string or null,
        "purchase_date": MM/DD/YY or null
    }}
    """
     
    res = llm.invoke([
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_input)
        ])
    
    complaint_data = json.loads(res.content)
     
    return {"complaint_data":complaint_data}

def create_ticket_node(state:SupervisorState)->SupervisorState:
    """
    Finalizes the complaint by validating entity specificity and generating a user confirmation number.
    """
    complaint_data = state.get("complaint_data",[])
    
    prefix = ''.join(random.choices(string.ascii_uppercase,k=2))
    number1 = random.randint(100,1000)
    mid = ''.join(random.choices(string.ascii_uppercase))
    number2 = random.randint(10,100)
    ticket_id = f"{prefix}{number1}{mid}{number2}"

    ticket={
        "ticket_id":ticket_id,
        "status":"created",
        "details":complaint_data
    }
    return {"messages":f"Ticket {ticket['ticket_id']} has been created."}

def ask_missing_node(state:SupervisorState)->SupervisorState:
    """ 
    Prompts the user for specific missing details to complete the support ticket data.
    """
    
    complaint = state.get("complaint_data", {})
    missing_info = state.get("missing_info",{})
    prompt = f"The user provided: {complaint}. Politely ask for the following missing information {missing_info}."
    response = llm.invoke(prompt)

    return {"messages": [response.content],
            "missing_fields": []}
  

In [ ]:
def router(state:SupervisorState)->SupervisorState:
    
    complaint_data=state.get("complaint_data",[])
    required_fields=["product", "issue_type", "purchase_date"]
    missing = [field for field in required_fields if not complaint_data.get(field)]

    if missing:
            return Command(
                goto="ask_missing_node", 
                update={
                    "missing_info": missing,
                    "complaint_data":complaint_data
                    } 
                )
    return Command(goto="create_ticket_node")
    

In [ ]:
def build_complaint_graph():
    
    complaint_graph = StateGraph(SupervisorState)
    #Add Nodes
    complaint_graph.add_node('extract_info_node', extract_info_node)
    complaint_graph.add_node("ask_missing_node", ask_missing_node)
    complaint_graph.add_node("create_ticket_node",create_ticket_node)
    complaint_graph.add_node("router", router)

    #Add Edges
    complaint_graph.add_edge(START,'extract_info_node')
    complaint_graph.add_edge("extract_info_node","router")
    complaint_graph.add_edge("ask_missing_node",END)
    complaint_graph.add_edge("create_ticket_node",END)
    memory = MemorySaver()
    return complaint_graph.compile(checkpointer=memory)

complaint_graph = build_complaint_graph()

def complaint_flow(state:SupervisorState)->SupervisorState:
    config = {"configurable": {"thread_id": "support_session_1"}}
    return complaint_graph.invoke(state, config=config)


In [ ]:
complaint_graph

In [ ]:
usr_state = { 
    "user_input": "Oct 2, 2025."
}

In [ ]:
complaint_flow(usr_state)

In [ ]:
def churn_score_node(state:SupervisorState)->SupervisorState:
    """
        Calculate a churn risk score from user input.
        Returns category: high, medium, or low.
    """
    user_input=state['user_input']
    current_profile = state.get('customer_profile',{})
    system_prompt = """ 
    You are a churn detection assistant.
    Based on the user input, classifiy their churn risk into:
    - high: user explicitly wants to cancel, switch, or sounds very frustrated.
    - medium: user shows dissatisfaction but hasn't decided to cancel yet.
    - low: user just asking questions or mild complaints.

    Examples:
    User: "I'm cancelling this useless service today."
    Churn risk: high

    User: "Your prices keep going up, I don't know if it's worth it anymore."
    Churn risk: medium

    User: "How do I cancel if I ever need to in the future?"
    Churn risk: low

    Respond with only one of: high, medium, low.
    """
    score = llm.invoke([
        {'role':'system', 'content':system_prompt},
        {'role':'user', 'content':user_input}
    ]).content
    
    current_profile['churn_score']=score
    return {"customer_profile":current_profile}

def loyalty_score_node(state:SupervisorState)->SupervisorState:
    """ 
        Provide loyalty score to user based on churn score and customer value. 
    """
    score = state['customer_profile']['churn_score']
    current_profile = state.get('customer_profile')
    reward_weights = {'high': 1.0, "medium": 0.6, "low": 0.2}
    clv_values = {"high": 1000, "medium": 500, "low": 200}
    clv_tier=random.choices(
        ['high','medium','low'],
        weights=[0.25,0.45,0.3],
        k=1
        )[0]

    loyalty_score = reward_weights[score]*clv_values[clv_tier]
    current_profile['loyalty_score']=loyalty_score
    current_profile['clv_tier']=clv_tier

    return {'customer_profile':current_profile}

def reward_node(state:SupervisorState)->SupervisorState:
    """
    Generate a personalized reward offers.
    """

    user_input=state['user_input']
    churn_score=state['customer_profile']['churn_score']
    loyalty_score=state['customer_profile']['loyalty_score']

    system_prompt=""" 
    You are a customer retention assistant. 
    Generate a polite, empathetic message based on the following:
    - The user's message
    - Their churn risk (high, medium, low)
    - Their loyalty score (40-1000)

    Rules:
    - If loyalty score >= 800 → emphasize strong appreciation and give a high reward (e.g., big discount, free premium month).
    - If 500-799 → show gratitude and offer a medium reward (e.g., discount or perk).
    - If 200-499 → acknowledge their value and give a small reward (e.g., loyalty points or small discount).
    - If < 200 → do not give a reward, just apologize and promise to improve.
    - If churn risk = high → always start by apologizing and showing empathy before mentioning any reward.
    - Keep the message short, friendly, concise and natural. Do not include technical terms or scores.
    
    Respond with only the final concise message.
    """

    user_context = f"""
        User message: {user_input}
        Churn risk: {churn_score}
        Loyalty score: {loyalty_score}
        """

    response = llm.invoke([{'role':'system', 'content':system_prompt},
                           {'role':'user','content':user_context}])
    
    return {"messages":response.content}


In [ ]:
def build_retention_graph():

    graph = StateGraph(SupervisorState)

    graph.add_node("churn_score",churn_score_node)
    graph.add_node("evaluate_user_value",loyalty_score_node)
    graph.add_node("offer_reward",reward_node)

    graph.add_edge(START, "churn_score")
    graph.add_edge("churn_score", "evaluate_user_value")
    graph.add_edge("evaluate_user_value", "offer_reward")
    graph.add_edge("offer_reward", END)
    memory=MemorySaver()
    return graph.compile(checkpointer=memory)

In [ ]:
retention_graph = build_retention_graph()

def retention_flow(state:SupervisorState)->SupervisorState:
    config= {"configurable":{"thread_id":"user_complaint_session", "recursion_limit":5}}
    return retention_graph.invoke(state, config=config)

In [ ]:
usr_input={"user_input":"I have problem with this phone. and you are the worst."}

In [ ]:
retention_flow(usr_input)

In [ ]:
def build_graph():

    graph = StateGraph(SupervisorState)

    graph.add_node("supervisor",supervisor_node)
    graph.add_node("greeting_flow",greeting_node)
    graph.add_node("clarification_flow", clarification_node)
    graph.add_node("complaint_flow", complaint_flow)
    graph.add_node("retention_flow", retention_flow)
    graph.add_node("inquiry_flow", inquiry_flow)

    graph.add_edge(START, "supervisor")
    graph.add_edge("greeting_flow", END)
    graph.add_edge("clarification_flow", END)

    memory = MemorySaver()
    return graph.compile(checkpointer=memory)

graph = build_graph()


In [ ]:
def agent(state:SupervisorState)->SupervisorState:
    config = {"configurable": {"thread_id": "support_session_1"}}
    return graph.invoke(state,config)

user_input = {"user_input":"I am having issue with my phone."}

In [ ]:
res = agent(user_input)

rprint(res)

In [ ]:
for msg in res["messages"]:
    print(msg)

In [ ]:
import torch
from transformers import pipeline
dialogue_classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small"  # fast, accurate, ~180MB
)

In [ ]:
DIALOGUE_LABELS = [
    "customer reporting something is broken, not working, or malfunctioning",
    "customer asking a question to get information or learn how to do something",
    "customer expressing unhappiness or requesting action like return or replacement",
    "social message like a greeting, goodbye, or unrelated conversation",
    "customer describing a situation without saying what they want done" 
]

In [ ]:
def classify_dialogue_act(message:str)->dict:
    result = dialogue_classifier(
        message,
        candidate_labels=DIALOGUE_LABELS,
        multi_label=False
        )
    return result['labels'][0], result['scores'][0]

In [ ]:
user_input="\
I want to close my account today." \
""

In [ ]:
res = classify_dialogue_act(user_input)

In [ ]:
res

In [ ]:
test_cases = [
    "My phone stopped working yesterday",
    "Screen is not working",
    "I want a replacement for my broken phone",
    "How do I reset my device",
    "I am switching to Samsung",
    "Hi there",
    "Thanks bye"
]

In [ ]:
for msg in test_cases:
    result = dialogue_classifier(msg, candidate_labels=DIALOGUE_LABELS)
    top_label = result["labels"][0]
    top_score = result["scores"][0]
    second_score = result["scores"][1]
    margin = top_score - second_score
    print(f"\n'{msg}'")
    print(f"  → {top_label[:50]}")
    print(f"  score: {top_score:.2f}  margin: {margin:.2f}")
